# Notebook 04 — Topic Modeling

## Purpose

Compare NMF, LDA, and BERTopic on all 5,292 comments. Manual topic labels are not a training target.

## Function words

The first run, `20261019_topic_models`, used no stopword list. Its selected topics were led by frequent grammatical words. This run removes the tokens in `config/somali_function_words.csv`. Every removed token is a grammatical word attested in at least 30 comments. Content words such as `lacag`, `nira`, `kaarka`, `dadka`, and `madaxweyne` stay in the model. The list is a project list for this corpus. It is not a published Somali dictionary.

## Selection rule

NMF and LDA use topic counts in {6, 8, 10, 12, 15}. A candidate is coverage-eligible when it assigns a topic to at least 80 percent of the 5,292 comments. The selected model is the coverage-eligible candidate with the highest `c_npmi`. If the leading scores are within 0.005, the higher diversity is selected. Comments with fewer than two remaining tokens are unassigned.

BERTopic reuses the multilingual MiniLM embeddings saved by the first run. Those embeddings do not show that the model understands Somali.

The first run remains on disk. This run is the one cited for topic terms.


## 1. Load the corpus and the function-word list

Fit topic models on all 5,292 comments. Manual topic labels are not an input.


In [1]:
from pathlib import Path
import json
import re
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

candidates = [Path.cwd(), Path.cwd().parent, Path(r"D:\NIRA_CASE_STUDY")]
ROOT = None
for candidate in candidates:
    if (candidate / "NIRA_Facebook_Datasets_5292.csv").exists():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    raise FileNotFoundError("Project root not found.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config.project_config import EXPECTED_RAW_N, RANDOM_SEED

RUN_IN = ROOT / "outputs" / "01_data_preparation" / "20261018_content_review"
PRIOR = ROOT / "outputs" / "04_topic_modeling" / "20261019_topic_models"
RUN_ID = "20261019_topic_models_function_words"
OUT = ROOT / "outputs" / "04_topic_modeling" / RUN_ID
FIG = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
np.random.seed(RANDOM_SEED)

TOKEN_RE = re.compile(r"(?u)\b[^\W\d_]{2,}\b")
PHONE_LIKE = re.compile(
    r"(?i)(?:\+?252[\s\-]?\d[\d\s\-]{6,}\d|\btell\s*:\s*\+?\d[\d\s\-]+|\b\d{10,15}\b)"
)
K_VALUES = [6, 8, 10, 12, 15]
TOP_N = 10
stops = pd.read_csv(ROOT / "config" / "somali_function_words.csv")
STOPWORDS = sorted(set(stops["word"].astype(str).str.lower()))
if len(STOPWORDS) < 20:
    raise AssertionError("The function-word list is unexpectedly short.")

raw = pd.read_csv(RUN_IN / "raw_corpus_working.csv").sort_values("source_row").reset_index(drop=True)
if len(raw) != EXPECTED_RAW_N or raw["record_key"].nunique() != EXPECTED_RAW_N:
    raise AssertionError("Working corpus size or keys are unexpected.")
texts = raw["comment_whitespace_normalized"].fillna("").astype(str)
embeddings = np.load(PRIOR / "comment_embeddings.npy")
if embeddings.shape[0] != len(raw):
    raise AssertionError("Saved embeddings do not match the corpus.")
print("Comments:", len(raw))
print("Function words removed:", len(STOPWORDS))
print("Output:", OUT)


Comments: 5292
Function words removed: 56
Output: D:\NIRA_CASE_STUDY\outputs\04_topic_modeling\20261019_topic_models_function_words


### Interpretation

All 5,292 comments are loaded in source-file order. Fifty-six function words are removed from the topic vocabulary. Content words such as `lacag`, `nira`, and `kaarka` remain. The embeddings are reused from the earlier run so this comparison uses the same vectors.


## 2. Tokenize the comments

Remove the 56 grammatical tokens listed for this project, then build the count and TF-IDF matrices.


In [2]:
def tokenize(text):
    return [tok for tok in TOKEN_RE.findall(str(text).lower()) if tok not in STOPWORDS]

def redact(text):
    return PHONE_LIKE.sub("[PHONE REDACTED]", str(text))

def topic_diversity(topic_words):
    pooled = [word for topic in topic_words for word in topic]
    return 0.0 if not pooled else len(set(pooled)) / len(pooled)

def coherence_score(topic_words, tokenized_docs, dictionary):
    cleaned = []
    for words in topic_words:
        keep = [word for word in words if word in dictionary.token2id]
        if len(keep) >= 2:
            cleaned.append(keep)
    if len(cleaned) < 2:
        return None
    value = float(CoherenceModel(
        topics=cleaned, texts=tokenized_docs, dictionary=dictionary,
        coherence="c_npmi", processes=1,
    ).get_coherence())
    return None if np.isnan(value) else value

def top_words_from_components(components, feature_names, n_words):
    names = []
    for row in components:
        order = np.argsort(row)[::-1][:n_words]
        names.append([feature_names[i] for i in order])
    return names

tokenized = [tokenize(text) for text in texts]
has_tokens = np.array([len(tokens) >= 2 for tokens in tokenized])
dictionary = Dictionary(tokenized)
dictionary.filter_extremes(no_below=5, no_above=0.6, keep_n=4000)
tokenized = [[word for word in doc if word in dictionary.token2id] for doc in tokenized]
vectorizer_kwargs = dict(
    token_pattern=r"(?u)\b[^\W\d_]{2,}\b",
    lowercase=True,
    min_df=5,
    max_df=0.6,
    max_features=4000,
    stop_words=STOPWORDS,
)
count_vectorizer = CountVectorizer(**vectorizer_kwargs)
tfidf_vectorizer = TfidfVectorizer(**vectorizer_kwargs)
count_matrix = count_vectorizer.fit_transform(texts)
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
count_features = np.array(count_vectorizer.get_feature_names_out())
tfidf_features = np.array(tfidf_vectorizer.get_feature_names_out())
print("Comments with at least two remaining tokens:", int(has_tokens.sum()))
print("Unassigned because fewer than two tokens remain:", int((~has_tokens).sum()))
print("TF-IDF vocabulary:", len(tfidf_features))


Comments with at least two remaining tokens: 5258
Unassigned because fewer than two tokens remain: 34
TF-IDF vocabulary: 1676


### Interpretation

After the function words are removed, 5,258 comments still have at least two tokens. Thirty-four comments become too short to assign. The TF-IDF vocabulary contains 1,676 terms. The unassigned comments stay in the 5,292 denominator.


## 3. Compare NMF and LDA

Fit both models at 6, 8, 10, 12, and 15 topics and score each one on the same comments.


In [3]:
rows = []
fitted = {}
for k in K_VALUES:
    nmf = NMF(n_components=k, init="nndsvda", random_state=RANDOM_SEED, max_iter=400)
    nmf_weights = nmf.fit_transform(tfidf_matrix)
    nmf_topics = top_words_from_components(nmf.components_, tfidf_features, TOP_N)
    assigned = np.full(len(raw), -1, dtype=int)
    scored = has_tokens & (np.asarray(tfidf_matrix.sum(axis=1)).ravel() > 0)
    assigned[scored] = nmf_weights[scored].argmax(axis=1)
    rows.append({
        "family": "NMF", "setting": f"k={k}", "n_topics": k,
        "coherence_c_npmi": coherence_score(nmf_topics, tokenized, dictionary),
        "topic_diversity": topic_diversity(nmf_topics),
        "coverage": float((assigned >= 0).mean()),
        "outlier_share": float((assigned < 0).mean()),
    })
    fitted[("NMF", k)] = {"topics": nmf_topics, "assigned": assigned, "weights": nmf_weights}
    print("NMF", k, rows[-1]["coherence_c_npmi"], round(rows[-1]["coverage"], 3))

    lda = LatentDirichletAllocation(
        n_components=k, random_state=RANDOM_SEED, learning_method="batch", max_iter=15
    )
    lda_doc = lda.fit_transform(count_matrix)
    lda_topics = top_words_from_components(lda.components_, count_features, TOP_N)
    assigned = np.full(len(raw), -1, dtype=int)
    scored = has_tokens & (np.asarray(count_matrix.sum(axis=1)).ravel() > 0)
    assigned[scored] = lda_doc[scored].argmax(axis=1)
    rows.append({
        "family": "LDA", "setting": f"k={k}", "n_topics": k,
        "coherence_c_npmi": coherence_score(lda_topics, tokenized, dictionary),
        "topic_diversity": topic_diversity(lda_topics),
        "coverage": float((assigned >= 0).mean()),
        "outlier_share": float((assigned < 0).mean()),
    })
    fitted[("LDA", k)] = {"topics": lda_topics, "assigned": assigned, "weights": lda_doc}
    print("LDA", k, rows[-1]["coherence_c_npmi"], round(rows[-1]["coverage"], 3))
comparison = pd.DataFrame(rows)
comparison.to_csv(OUT / "topic_model_grid_bow.csv", index=False)


NMF 6 0.012451188227955295 0.959


LDA 6 0.022543746769311743 0.959


NMF 8 -0.028841576716568926 0.959


LDA 8 0.004991149142354656 0.959


NMF 10 -0.013491959796369265 0.959


LDA 10 -0.02842518021873015 0.959


NMF 12 -0.030052839607794122 0.959


LDA 12 -0.043498206008522035 0.959


NMF 15 -0.06467208422312629 0.959


LDA 15 -0.04704509127866544 0.959


### Interpretation

NMF and LDA both assign a topic to 95.9 percent of comments. LDA with 6 topics has the highest coherence in this grid, 0.023. Several larger settings have negative coherence. A higher topic count is not treated as a better model.


## 4. Fit BERTopic

Reuse the saved multilingual embeddings. The embedding model is not evidence that the representation understands Somali.


In [4]:
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
bert_vectorizer = CountVectorizer(**vectorizer_kwargs)
bertopic_model = BERTopic(
    embedding_model=embedder,
    umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=RANDOM_SEED),
    hdbscan_model=HDBSCAN(min_cluster_size=30, min_samples=5, metric="euclidean", prediction_data=True),
    vectorizer_model=bert_vectorizer,
    top_n_words=TOP_N,
    calculate_probabilities=False,
    verbose=False,
)
bert_topics, _ = bertopic_model.fit_transform(texts.tolist(), embeddings=embeddings)
bert_assigned = np.array(bert_topics, dtype=int)
bert_assigned[~has_tokens] = -1
topic_ids = sorted(topic_id for topic_id in set(bert_assigned.tolist()) if topic_id >= 0)
bert_topic_words = []
for topic_id in topic_ids:
    pairs = bertopic_model.get_topic(topic_id) or []
    bert_topic_words.append([word for word, _score in pairs][:TOP_N])
bert_row = {
    "family": "BERTopic",
    "setting": "min_cluster_size=30; function words removed; saved MiniLM embeddings",
    "n_topics": int(len(topic_ids)),
    "coherence_c_npmi": coherence_score(bert_topic_words, tokenized, dictionary) if len(bert_topic_words) >= 2 else None,
    "topic_diversity": topic_diversity(bert_topic_words),
    "coverage": float((bert_assigned >= 0).mean()),
    "outlier_share": float((bert_assigned < 0).mean()),
}
fitted[("BERTopic", "auto")] = {"topics": bert_topic_words, "assigned": bert_assigned, "topic_ids": topic_ids}
print(bert_row)


C:\Users\xaale\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4979.81it/s]

{'family': 'BERTopic', 'setting': 'min_cluster_size=30; function words removed; saved MiniLM embeddings', 'n_topics': 34, 'coherence_c_npmi': -0.30529403401589333, 'topic_diversity': 0.7647058823529411, 'coverage': 0.4914965986394558, 'outlier_share': 0.5085034013605442}


### Interpretation

BERTopic returns 34 topics, but it leaves about half of the comments unassigned. Its coherence is negative. Under the pre-set rule, a model must cover at least 80 percent of the 5,292 comments before it can be selected.


## 5. Select one model and save its topics

Apply the coverage rule that was fixed before the scores were read.


In [5]:
full_comparison = pd.concat([comparison, pd.DataFrame([bert_row])], ignore_index=True)
full_comparison.to_csv(OUT / "topic_model_comparison.csv", index=False)
eligible = full_comparison.dropna(subset=["coherence_c_npmi"])
eligible = eligible[eligible["coverage"] >= 0.80]
if eligible.empty:
    pool = full_comparison.dropna(subset=["coherence_c_npmi"]).copy()
    coverage_rule = "No candidate assigned a topic to at least 80 percent of comments. The highest coherence was selected, and the coverage shortfall is part of the result."
else:
    pool = eligible.copy()
    coverage_rule = "The selected model is the coverage-eligible candidate with the highest c_npmi. Coverage-eligible means a topic was assigned to at least 80 percent of the 5,292 comments."
pool = pool.sort_values(["coherence_c_npmi", "topic_diversity", "coverage"], ascending=[False, False, False]).reset_index(drop=True)
winner = pool.iloc[0]
if len(pool) > 1 and abs(float(pool.loc[0, "coherence_c_npmi"]) - float(pool.loc[1, "coherence_c_npmi"])) <= 0.005:
    winner = pool.iloc[:2].sort_values(["topic_diversity", "coverage"], ascending=[False, False]).iloc[0]
    coverage_rule += " The leading coherence values were within 0.005, so the higher diversity decided the tie."
chosen_key = ("BERTopic", "auto") if winner["family"] == "BERTopic" else (winner["family"], int(winner["n_topics"]))
chosen = fitted[chosen_key]
assigned = chosen["assigned"]
term_lookup = {}
if winner["family"] == "BERTopic":
    for topic_id, words in zip(chosen["topic_ids"], chosen["topics"]):
        term_lookup[int(topic_id)] = words
else:
    for topic_id, words in enumerate(chosen["topics"]):
        term_lookup[int(topic_id)] = words
assignments = pd.DataFrame({
    "source_row": raw["source_row"].to_numpy(),
    "record_key": raw["record_key"].to_numpy(),
    "post_id": raw["post_id"].to_numpy(),
    "topic_id": assigned,
})
assignments["topic_terms"] = assignments["topic_id"].map(
    lambda topic_id: " | ".join(term_lookup.get(int(topic_id), [])) if int(topic_id) >= 0 else ""
)
assignments["assignment_status"] = np.where(assignments["topic_id"] >= 0, "assigned", "unassigned")
assignments.to_csv(OUT / "topic_assignments.csv", index=False)
topic_rows = []
for topic_id, words in term_lookup.items():
    topic_rows.append({
        "topic_id": int(topic_id),
        "n_comments": int((assigned == topic_id).sum()),
        "top_terms": " | ".join(words),
        "provisional_term_label": " | ".join(words[:3]),
    })
topic_table = pd.DataFrame(topic_rows).sort_values("n_comments", ascending=False)
topic_table.to_csv(OUT / "selected_topic_terms.csv", index=False)
rank_matrix = None if winner["family"] == "BERTopic" else chosen["weights"]
samples = []
for topic_id in topic_table["topic_id"]:
    member_index = np.where(assigned == int(topic_id))[0]
    if rank_matrix is not None:
        scores = rank_matrix[member_index, int(topic_id)]
        order = member_index[np.argsort(scores)[::-1][:3]]
    else:
        centroid = embeddings[member_index].mean(axis=0)
        centroid = centroid / max(np.linalg.norm(centroid), 1e-8)
        scores = embeddings[member_index] @ centroid
        order = member_index[np.argsort(scores)[::-1][:3]]
    for rank, idx in enumerate(list(order), start=1):
        samples.append({
            "topic_id": int(topic_id),
            "rank_in_topic": rank,
            "record_key": raw.loc[idx, "record_key"],
            "post_id": raw.loc[idx, "post_id"],
            "excerpt": redact(raw.loc[idx, "comment_original"])[:240],
        })
pd.DataFrame(samples).to_csv(OUT / "selected_topic_examples.csv", index=False)
decision = {
    "selected_family": str(winner["family"]),
    "selected_setting": str(winner["setting"]),
    "n_topics": int(winner["n_topics"]),
    "coherence_c_npmi": None if pd.isna(winner["coherence_c_npmi"]) else float(winner["coherence_c_npmi"]),
    "topic_diversity": float(winner["topic_diversity"]),
    "coverage": float(winner["coverage"]),
    "outlier_share": float(winner["outlier_share"]),
    "unassigned_comments": int((assigned < 0).sum()),
    "comments_in_corpus": int(len(raw)),
    "function_words_removed": len(STOPWORDS),
    "function_word_file": "config/somali_function_words.csv",
    "prior_run": "20261019_topic_models",
    "selection_rule": coverage_rule,
    "manual_labels_used_as_training_target": False,
    "language_limitation": "Top terms are Somali word forms from a statistical model after removal of the project function-word list. Coherence does not confirm a public-policy theme. The embedding model is multilingual and is not evidence of Somali understanding.",
}
(OUT / "topic_model_selection.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")
stops.to_csv(OUT / "function_words_used.csv", index=False)
print(json.dumps(decision, indent=2))
print(topic_table.to_string(index=False))
plot_df = full_comparison.dropna(subset=["coherence_c_npmi"]).copy()
plot_df["label"] = plot_df["family"] + " " + plot_df["setting"].astype(str)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_df["label"], plot_df["coherence_c_npmi"])
ax.set_xlabel("c_npmi coherence")
ax.set_title("Topic-model coherence after function-word removal")
fig.tight_layout()
fig.savefig(FIG / "topic_coherence_comparison.png", dpi=150)
plt.close(fig)
(PRIOR / "SUPERSEDED_BY_FUNCTION_WORD_RUN.txt").write_text(
    "Topic terms from this run are retained for comparison. Cite 20261019_topic_models_function_words for topic terms. The embedding file in this folder is still the embedding source.\n",
    encoding="utf-8",
)


{
  "selected_family": "LDA",
  "selected_setting": "k=6",
  "n_topics": 6,
  "coherence_c_npmi": 0.022543746769311743,
  "topic_diversity": 0.7833333333333333,
  "coverage": 0.9591836734693877,
  "outlier_share": 0.04081632653061224,
  "unassigned_comments": 216,
  "comments_in_corpus": 5292,
  "function_words_removed": 56,
  "function_word_file": "config/somali_function_words.csv",
  "prior_run": "20261019_topic_models",
  "selection_rule": "The selected model is the coverage-eligible candidate with the highest c_npmi. Coverage-eligible means a topic was assigned to at least 80 percent of the 5,292 comments.",
  "manual_labels_used_as_training_target": false,
  "language_limitation": "Top terms are Somali word forms from a statistical model after removal of the project function-word list. Coherence does not confirm a public-policy theme. The embedding model is multilingual and is not evidence of Somali understanding."
}
 topic_id  n_comments                                           

179

### Interpretation

LDA with 6 topics is selected. It covers 95.9 percent of comments and leaves 216 unassigned. Its coherence is 0.023 and its term diversity is 0.783. The leading terms include `lacag`, `kaarka`, `nira`, `dadka`, and `madaxweyne`. These are the model's terms, not validated names of policy themes. BERTopic is reported in the comparison and is not selected.
